# **qlora_v1.ipynb**

PEFT-QLoRA example for fine-tuning with the FreedomIntelligence/medical-o1-reasoning-SFT dataset. This code is inspired by community projects and uses the lightweight DeepSeek-R1-Distill-Qwen-1.5B model, making it efficient even on a free T4 GPU.

**How It Works**

**Dataset Formatting**: The medical-o1-reasoning-SFT dataset contains Question, Complex_CoT, and Response fields. The code formats these into a chat template that encourages step-by-step reasoning (<think>...</think>) before the final answer .

**Efficient Training**: QLoRA dramatically reduces memory usage by quantizing the base model to 4-bit, allowing this 1.5B parameter model to be trained on a standard T4 GPU . The SFTTrainer from the trl library handles the supervised fine-tuning loop


**bitsandbytes** - The 4-bit **Quantization Engine
Purpose: Enables 4-bit quantization to dramatically reduce model memory usage.

**Why You Need It**:

Compresses a 1.5B parameter model from ~6GB (FP16) down to ~1.5GB (4-bit)

Allows training on free Colab GPUs (T4 with 16GB VRAM)

**Libraries Used:**

**trl - Transformer Reinforcement Learning**
Purpose: Provides high-level trainers and utilities for fine-tuning language models, especially with RLHF and instruction tuning.

Why You Need It:

Provides SFTTrainer - the most convenient way to fine-tune on instruction datasets

Handles tokenization, batching, and training loop automatically

Built specifically for conversational/chat models

In [ ]:
# 1. Install required libraries
!pip install -q accelerate peft transformers datasets trl bitsandbytes
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset
import torch

# 2. Configure model & QLoRA (4-bit quantization)
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
# NF4 (Normal Float 4): Optimal 4-bit quantization for LLMs
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [ ]:
# 3. Load and format the datase
dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", name="en", split="train[:5000]")  # Use subset for speed, 50, 500 and then remove it

In [ ]:
def format_chat_template(example):
    return f"<|user|>\n{example['Question']}<|end|>\n<|assistant|>\n<think>\n{example['Complex_CoT']}\n</think>\n{example['Response']}"

In [ ]:
# 4. Configure LoRA adapters
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = prepare_model_for_kbit_training(model) # Reintroduced this line
# model = get_peft_model(model, lora_config) # This remains removed, as SFTTrainer handles it internally

In [ ]:
# 5. Train with SFTTrainer
training_args = TrainingArguments(
    output_dir="./medical-qlora-results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    save_steps=50,
    logging_steps=25,
    learning_rate=2e-4,
    warmup_steps=2, # Changed from warmup_ratio=0.03
    fp16=False,
    bf16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    peft_config=lora_config, # Added peft_config here
    formatting_func=format_chat_template,
)

trainer.train()
# 6. Save the fine-tuned adapter
model.save_pretrained("./medical-qlora-adapter")
print("✅ Fine-tuning complete. LoRA adapter saved.")

Applying formatting function to train dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 151646, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Fine-tuning complete. LoRA adapter saved.
